In [ ]:
# ============================================
# MLflow Model Serving - Train, Log, Serve, Predict
# Dataset: Breast Cancer (sklearn built-in)
# ============================================

# Step 1: Import Libraries
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from urllib.parse import urlparse
import numpy as np
import pandas as pd

mlflow.autolog()

# Step 2: Load and Prepare Data
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# Step 3: Train and Log the Model
with mlflow.start_run():
    lr = LogisticRegression(max_iter=10000, random_state=42)
    lr.fit(X_train, y_train)

    predictions = lr.predict(X_train)
    signature = infer_signature(X_train, predictions)

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    if tracking_url_type_store != "file":
        mlflow.sklearn.log_model(
            lr, "model", registered_model_name="BreastCancerModel", signature=signature
        )
    else:
        mlflow.sklearn.log_model(lr, "model", signature=signature)

    acc = accuracy_score(y_test, lr.predict(X_test))
    print(f"Test Accuracy: {acc:.4f}")

# Step 4: To serve the model, run this in a SEPARATE terminal:
# mlflow models serve --env-manager=local -m mlruns/0/<RUN_ID>/artifacts/model -h 127.0.0.1 -p 5001
print("\nTo serve the model, open a new terminal and run:")
print("mlflow models serve --env-manager=local -m mlruns/0/<RUN_ID>/artifacts/model -h 127.0.0.1 -p 5001")
print("Replace <RUN_ID> with the actual run ID from mlruns folder")

# Step 5: After serving, run this to make predictions via HTTP
print("\n--- After serving, run the code below to test predictions ---")
print(f"Test data shape: {X_test.shape}")

# Uncomment below after starting the server:
# import requests
# import json
# url = 'http://127.0.0.1:5001/invocations'
# data_payload = {
#     "columns": list(feature_names),
#     "instances": X_test.tolist()
# }
# response = requests.post(url, json=data_payload)
# predictions = response.json()
# print(predictions)